# Movie Prediction Based on Cosine Similarity

In [ ]:
!wget https://files.grouplens.org/datasets/movielens/ml-32m.zip

In [ ]:
!unzip ml-32m.zip
!rm ml-32m.zip
import pandas as pd


In [ ]:
df_ratings = pd.read_csv('ml-32m/ratings.csv')
df_tags = pd.read_csv('ml-32m/tags.csv')
df_movies = pd.read_csv('ml-32m/movies.csv')

In [ ]:
df_ratings.head()

In [ ]:
df_tags.head()

In [ ]:
df_movies.head()

In [ ]:
movie_rated_mean = df_ratings.groupby("movieId").rating.mean()

In [ ]:
df_movies = df_movies.merge(movie_rated_mean, left_on="movieId", right_on="movieId")

In [ ]:
df_movies.head()

In [ ]:
df_tags.head()

In [ ]:
# Group df_tags by movieId and aggregate tags into a list
movie_tags = df_tags.groupby('movieId')['tag'].apply(list).reset_index()

In [ ]:
# Group df_tags by movieId and aggregate tags into a list
movie_tags = df_tags.groupby('movieId')['tag'].apply(list).reset_index()

# Merge the aggregated tags into df_movies
df_movies = df_movies.merge(movie_tags, on='movieId', how='left')

# Display the updated df_movies DataFrame
df_movies.head()

# Task
Find 5 similar movies for a given movie based on their genres and tags.

## Feature engineering

### Subtask:
Create a combined feature representation for each movie using its genres and tags. This might involve techniques like one-hot encoding for genres and potentially using techniques like TF-IDF for tags.


**Reasoning**:
Handle missing values in the 'tag' column and create a combined string of genres and tags.



In [ ]:
df_movies['tag'] = df_movies['tag'].fillna('').astype(str)
df_movies['combined_features'] = df_movies['genres'] + ' ' + df_movies['tag']

**Reasoning**:
Initialize and apply TF-IDF vectorization to the combined features.



In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df_movies['combined_features'])

## Similarity calculation

### Subtask:
Calculate the similarity between movies based on their feature representations using cosine similarity.


**Reasoning**:
Calculate the cosine similarity between movies based on their combined features.



In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

## Recommendation function

### Subtask:
Create a function that takes a movie title as input and returns a list of the most similar movies based on the calculated similarity scores.


**Reasoning**:
Define the `get_recommendations` function to find similar movies based on the cosine similarity matrix.



In [ ]:
def get_recommendations(title, cosine_sim=cosine_sim):
    """
    Gets movie recommendations based on cosine similarity.

    Args:
        title: The title of the movie.
        cosine_sim: The cosine similarity matrix.

    Returns:
        A list of recommended movie titles, or a message if the movie is not found.
    """
    # Get the index of the movie that matches the title
    if title not in df_movies['title'].values:
        return f"Movie with title '{title}' not found in the dataset."

    idx = df_movies[df_movies['title'] == title].index[0]

    # Get the pairwise similarity scores for the movie
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort the movies based on the similarity scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get the scores of the 5 most similar movies (excluding the movie itself)
    sim_scores = sim_scores[1:6]

    # Get the movie indices
    movie_indices = [i[0] for i in sim_scores]

    # Return the top 5 most similar movies
    return df_movies['title'].iloc[movie_indices].tolist()

## Summary:

### Data Analysis Key Findings

*   Missing values in the 'tag' column were successfully handled by replacing them with empty strings.
*   A combined feature representation for each movie was created by concatenating the 'genres' and 'tag' columns.
*   A TF-IDF matrix was successfully generated from the combined features, representing the movies in a vector space.
*   The cosine similarity matrix between movies was calculated using the TF-IDF matrix.
*   A function `get_recommendations` was successfully created to return the top 5 most similar movies for a given movie title based on the cosine similarity scores.


In [ ]:
get_recommendations("Father of the Bride Part II (1995)")

In [ ]:
import re

def find_movie_title(title):
    """
    Finds a movie title in the df_movies DataFrame by focusing on the core content of the title,
    ignoring information before or after it, and handling variations in spacing and capitalization.

    Args:
        title: The movie title to search for.

    Returns:
        The exact movie title from the DataFrame if found, otherwise None.
    """
    # Function to extract the core title content using a regular expression
    def extract_core_title(t):
        # This regex attempts to capture the main title part. It might need adjustment
        # based on the specific patterns in your titles.
        match = re.search(r'^(.*?)\s*\(\d{4}\)', t) # Matches anything before a space followed by (year)
        if match:
            return match.group(1).strip().lower()
        return t.strip().lower() # Fallback to just stripping and lowercasing if pattern not found

    cleaned_input_title = extract_core_title(title)

    for movie_title in df_movies['title']:
        if extract_core_title(movie_title) == cleaned_input_title:
            return movie_title
    return None

# Examples found after cleaning:
found_title = find_movie_title("Toy Story")
print(found_title)
found_title = find_movie_title("  Good Will Hunting (1997)  ")
print(found_title)
found_title = find_movie_title("orca")
print(found_title)

In [ ]:
df_movies

In [ ]:
import re

def extract_year(title):
    """
    Extracts the four-digit year from a movie title string, assuming it's in parentheses at the end.

    Args:
        title: The movie title string.

    Returns:
        The extracted year as a string, or None if no year is found.
    """
    match = re.search(r'\((\d{4})\)$', title)
    if match:
        return match.group(1)
    return None

df_movies['year'] = df_movies['title'].apply(extract_year)

# Display the first few rows with the new 'year' column
display(df_movies.head())

In [ ]:
df_movies.year = pd.to_datetime(df_movies['year'], format='%Y')

In [ ]:
# Filter for movies released in the year 2023
top_movies = df_movies[df_movies['year'].dt.year == 2023].rating.sort_values().tail().index

In [ ]:
df_movies.iloc[top_movies]

In [ ]:
get_recommendations(find_movie_title("Remember the titans"))

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import re

# Create a text input widget
movie_title_input = widgets.Text(
    value='',
    placeholder='Enter movie title here',
    description='Movie Title:',
    disabled=False
)

# Function to provide movie title suggestions based on the input prefix
def get_suggestions(prefix, num_suggestions=5):
    """Provides movie title suggestions based on the input prefix."""
    cleaned_prefix = prefix.strip().lower()
    if not cleaned_prefix:
        return []

    suggestions = []
    for title in df_movies['title']:
        # Clean the movie title for comparison (remove year and clean)
        cleaned_title = re.sub(r'\s*\(\d{4}\)\s*$', '', title).strip().lower()
        if cleaned_prefix in cleaned_title:
            suggestions.append(title)

    # Return the top N suggestions
    return suggestions[:num_suggestions]

# Function to get and display recommendations
def get_and_display_recommendations(title):
    with output_widget:
        clear_output()
        if title:
            # Use the find_movie_title function to get the exact title
            exact_title = find_movie_title(title)
            if exact_title:
                recommendations = get_recommendations(exact_title)
                if isinstance(recommendations, list):
                    print(f"Recommendations for '{exact_title}':")
                    for movie in recommendations:
                        print(f"- {movie}")
                else:
                    # This handles the case where get_recommendations returns a "not found" message
                    print(recommendations)
            else:
                print(f"Movie title '{title}' not found or could not be matched.")
        else:
            print("Please enter a movie title.")

# Function to be called when an input suggestion button is clicked
def on_suggestion_button_clicked(b):
    get_and_display_recommendations(b.description)


# Function to be called when the input changes (for suggestions)
def handle_input_change(change):
    with output_widget:
        clear_output()
        entered_text = change['new']
        if entered_text:
            suggestions = get_suggestions(entered_text)
            if suggestions:
                print("Suggestions:")
                # Display suggestions as clickable buttons
                suggestion_buttons = [widgets.Button(description=s, layout=widgets.Layout(width='auto')) for s in suggestions]
                for btn in suggestion_buttons:
                    btn.on_click(on_suggestion_button_clicked)
                display(widgets.VBox(suggestion_buttons)) # Use VBox to arrange buttons vertically
            else:
                print("No suggestions found.")

# Create an output widget to display suggestions and recommendations
output_widget = widgets.Output()

# Observe changes in the input widget
movie_title_input.observe(handle_input_change, names='value')

# Display the widgets
display(movie_title_input, output_widget)

In [ ]:
!jupyter nbconvert --clear-output --inplace MoviePrediction.ipynb


In [ ]:
import nbformat

# Replace with your notebook filename if you've uploaded it
notebook_file = "MoviePrediction.ipynb"

# Read notebook
with open(notebook_file, "r", encoding="utf-8") as f:
    nb = nbformat.read(f, as_version=4)

# Drop broken widget metadata
if "widgets" in nb["metadata"]:
    nb["metadata"].pop("widgets")

# Save fixed notebook
fixed_file = notebook_file.replace(".ipynb", "_fixed.ipynb")
with open(fixed_file, "w", encoding="utf-8") as f:
    nbformat.write(nb, f)

print(f"Fixed notebook saved as {fixed_file}")
